# Data Extraction

This notebook extracts structured tabular data from the raw JSON files collected in `01_collection.ipynb`. Each raw file contains a GeoJSON response from the IEM Convective Outlook Warning (COW) API for a single NWS Weather Forecast Office and calendar year. The extraction process flattens those responses into two analysis-ready tables: `events`, containing one row per warning event, and `stormreports`, containing one row per Local Storm Report (LSR). Both tables are written as CSV files to `data/extracted/`, which serves as an immutable checkpoint of the source data before any cleaning or transformation is applied. No analytical decisions are made here; the output reflects the content of the source data as faithfully as possible.

## Implementation

Extraction is implemented by `COWExtractor` in `src/processing/extractor.py`. It reads every `{WFO}_{YEAR}.json` file from `data/raw/COW/`, flattens the GeoJSON feature properties into rows, and writes two CSV files to `data/extracted/` as an immutable checkpoint before any cleaning decisions are applied.

| Method | Output |
|---|---|
| `extract_events()` | One row per warning event; `wfo` and `year` derived from filename |
| `extract_stormreports()` | One row per LSR; `warned=False` rows are unwarned storm reports (misses) |

Geometry is excluded from both tables because the analysis is statistical, not spatial. API-supplied `wfo` and `year` fields are replaced by filename-derived values to ensure consistency across all files.

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("../src")

from processing import COWExtractor

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Input: raw JSON files written by 01_collection.ipynb
COW_DIR = Path("../data/raw/COW")

# Output: flattened CSV files, immutable checkpoint before cleaning.
# Cleaning decisions are applied in 03_cleaning.ipynb → data/processed/
EXTRACTED_DIR = Path("../data/extracted")

## Events

One row per warning event issued by a WFO. Key fields for analysis: `phenomena`, `fcster`, `issue`, `expire`, `verify`, `lead0`, and `significance`. The `stormreports` and `lead0` columns are null for unverified events (i.e., those with no matching LSR).

In [2]:
extractor = COWExtractor(raw_dir=COW_DIR, extracted_dir=EXTRACTED_DIR)
print(extractor)

COWExtractor(raw=../data/raw/COW, extracted=../data/extracted)


In [3]:
events = extractor.extract_events()
print(f"{len(events):,} rows × {len(events.columns)} columns")
events.head()

15:18:04  INFO     Extracting events from ABQ_2020.json ...
15:18:04  INFO     Extracting events from ABQ_2021.json ...
15:18:04  INFO     Extracting events from ABQ_2022.json ...
15:18:04  INFO     Extracting events from ABQ_2023.json ...
15:18:04  INFO     Extracting events from ABQ_2024.json ...
15:18:04  INFO     Extracting events from ABQ_2025.json ...
15:18:04  INFO     Extracting events from ABR_2020.json ...
15:18:04  INFO     Extracting events from ABR_2021.json ...
15:18:04  INFO     Extracting events from ABR_2022.json ...
15:18:04  INFO     Extracting events from ABR_2023.json ...
15:18:04  INFO     Extracting events from ABR_2024.json ...
15:18:04  INFO     Extracting events from ABR_2025.json ...
15:18:04  INFO     Extracting events from AFC_2020.json ...
15:18:04  WARNING  AFC_2020.json: no events found, skipping
15:18:04  INFO     Extracting events from AFC_2021.json ...
15:18:04  WARNING  AFC_2021.json: no events found, skipping
15:18:04  INFO     Extracting events fro

16,086 rows × 32 columns


,wfo,year,phenomena,eventid,sharedborder,issue,expire,statuses,fcster,svr_tornado_possible,...,stormreports,stormreports_all,verify,tor_in_svrtorpossible,lead0,areaverify,visual_imgurl,product_text,product_href,link
0,ABQ,2020,TO,1,40531.482739,2020-03-13T23:04:00Z,2020-03-13T23:30:00Z,EXP,DPorter,False,...,,,False,False,None,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...
1,ABQ,2020,TO,2,0.000000,2020-03-13T23:20:00Z,2020-03-13T23:32:00Z,CAN,DPorter,False,...,,,False,False,None,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...
2,ABQ,2021,TO,1,0.000000,2021-05-17T19:22:00Z,2021-05-17T19:45:00Z,EXP,44,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2021-...
3,ABQ,2021,TO,2,18209.013334,2021-05-17T22:42:00Z,2021-05-17T23:20:00Z,CAN,44,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2021-...
4,ABQ,2021,TO,3,47094.091641,2021-05-17T23:41:00Z,2021-05-17T23:55:00Z,CAN,44,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2021-...


In [4]:
extractor.save(events, "events.csv")

15:18:05  INFO     Saved 16,086 rows to ../data/extracted/events.csv


PosixPath('../data/extracted/events.csv')

## Storm Reports

One row per Local Storm Report (LSR). The `warned` boolean indicates whether a warning was in effect at the time of the report. Warned reports link back to the events table via the `events` foreign key column; unwarned reports (`warned=False`) have null `events` and `leadtime` values and represent misses in POD calculations.

In [5]:
stormreports = extractor.extract_stormreports()
print(f"{len(stormreports):,} rows × {len(stormreports.columns)} columns")
stormreports.head()

15:18:05  INFO     Extracting stormreports from ABQ_2020.json ...
15:18:05  INFO     Extracting stormreports from ABQ_2021.json ...
15:18:05  INFO     Extracting stormreports from ABQ_2022.json ...
15:18:05  INFO     Extracting stormreports from ABQ_2023.json ...
15:18:05  INFO     Extracting stormreports from ABQ_2024.json ...
15:18:05  INFO     Extracting stormreports from ABQ_2025.json ...
15:18:05  INFO     Extracting stormreports from ABR_2020.json ...
15:18:05  INFO     Extracting stormreports from ABR_2021.json ...
15:18:05  INFO     Extracting stormreports from ABR_2022.json ...
15:18:05  INFO     Extracting stormreports from ABR_2023.json ...
15:18:05  INFO     Extracting stormreports from ABR_2024.json ...
15:18:05  INFO     Extracting stormreports from ABR_2025.json ...
15:18:05  INFO     Extracting stormreports from AFC_2020.json ...
15:18:05  WARNING  AFC_2020.json: no stormreports found, skipping
15:18:05  INFO     Extracting stormreports from AFC_2021.json ...
15:18:05  

10,708 rows × 19 columns


,wfo,year,valid,type,magnitude,city,county,state,source,remark,typetext,lon0,lat0,events,tdq,warned,leadtime,lsrtype,link
0,ABQ,2020,2020-03-17T23:24:00Z,T,None,PLEASANT HILL,CURRY,NM,SOCIAL MEDIA,NaN,TORNADO,-103.07,34.52,,False,False,None,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
1,ABQ,2020,2020-03-21T20:30:00Z,T,None,6 S KIRTLAND,SAN JUAN,NM,PUBLIC,LANDSPOUT TORNADO OVER NAVAJO AGRICULTURAL PRO...,TORNADO,-108.34,36.65,,False,False,None,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
2,ABQ,2020,2020-07-04T22:30:00Z,T,None,13 W GRENVILLE,UNION,NM,STORM CHASER,NaN,TORNADO,-103.85,36.59,,False,False,None,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
3,ABQ,2020,2020-07-15T23:03:00Z,T,None,3 E VEGUITA,SOCORRO,NM,NWS EMPLOYEE,LANDSPOUT OCCURRED FOR ABOUT 15 MINUTES.,TORNADO,-106.71,34.52,,False,False,None,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
4,ABQ,2021,2021-01-25T14:15:00Z,T,None,4 NNE CHACON,MORA,NM,TRAINED SPOTTER,NaN,TORNADO,-105.35,36.20,,False,False,NaN,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...


In [6]:
extractor.save(stormreports, "stormreports.csv")

15:18:06  INFO     Saved 10,708 rows to ../data/extracted/stormreports.csv


PosixPath('../data/extracted/stormreports.csv')